# teacher(PMRiskViT) 학습 — `pm_safeline`

로드뷰 이미지 -> 사고 위험도 teacher 모델(ViT) 학습 노트북(PROJECT.md §4.4~§4.5).

- 데이터: `datasets.roadview.make_train_valid` (manifest.csv 필요)
- 모델: `models.pm_risk_vit.build_pm_risk_vit`
- 학습: `utils.training.train_teacher` (BCEWithLogitsLoss + pos_weight, early stopping on valid AUC)
- 신뢰도: `utils.training.cross_validate` (k-fold, §4.5-1 — 데이터 희소 → 고정 test 대신 CV)
- 보정: `utils.training.calibrate` (temperature scaling, §4.5 과신 보정)

> torch/torchvision 이 설치되지 않은 환경(예: 기본 Py3.14 venv)에서는 각 셀이 안내 메시지를 출력하고
> 건너뛴다. 학습을 실제로 돌리려면 torch 가 설치된 별도 env 를 사용하라.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath("../../main/python"))
os.environ.setdefault("PM_SV_PROVIDER", "mock")

try:
    import torch  # noqa: F401
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

if not HAS_TORCH:
    print(
        "[note] torch 가 설치되어 있지 않습니다. 이 노트북의 학습 셀은 안내만 출력하고 건너뜁니다.\n"
        "       학습을 실제로 돌리려면 torch/torchvision 이 설치된 env 를 사용하세요. 예:\n"
        "       `uv venv --python 3.12 .venv-train && uv pip install torch torchvision scikit-learn`"
    )
else:
    print(f"torch {torch.__version__} 감지됨 — 학습 셀 실행 가능")


[note] torch 가 설치되어 있지 않습니다. 이 노트북의 학습 셀은 안내만 출력하고 건너뜁니다.
       학습을 실제로 돌리려면 torch/torchvision 이 설치된 env 를 사용하세요. 예:
       `uv venv --python 3.12 .venv-train && uv pip install torch torchvision scikit-learn`


## 1. 데이터셋 구성 — `make_train_valid`

`data/manifest.csv` 가 없으면 먼저 수집 파이프라인을 실행해야 한다:
`python -m pm_safeline collect` (KoROAD 이륜차 사고 데이터 기준).

In [2]:
from pathlib import Path

MANIFEST_PATH = Path("../../../data/manifest.csv")

if not HAS_TORCH:
    print("[skip] torch 미설치 — 데이터셋 구성을 건너뜁니다.")
elif not MANIFEST_PATH.exists():
    print(
        f"[note] manifest 가 없습니다: {MANIFEST_PATH.resolve()}\n"
        "       먼저 `python -m pm_safeline collect` 로 데이터를 수집하세요."
    )
    train_ds = valid_ds = None
else:
    from pm_safeline.datasets.roadview import make_train_valid

    train_ds, valid_ds = make_train_valid(valid_frac=0.2, seed=42)
    print(f"train {len(train_ds)} / valid {len(valid_ds)} 샘플")


[skip] torch 미설치 — 데이터셋 구성을 건너뜁니다.


## 2. 모델 — `build_pm_risk_vit`

기본은 backbone 고정 + 헤드만 학습(linear probe, §4.5-1: 데이터 희소 대응).

In [3]:
if not HAS_TORCH:
    print("[skip] torch 미설치 — 모델 생성을 건너뜁니다.")
else:
    from pm_safeline.models.pm_risk_vit import build_pm_risk_vit, PMRiskViTConfig

    model = build_pm_risk_vit(PMRiskViTConfig(freeze_backbone=True))
    n_trainable = sum(p.numel() for p in model.trainable_parameters())
    print(f"학습 가능 파라미터 수: {n_trainable:,}")


[skip] torch 미설치 — 모델 생성을 건너뜁니다.


## 3. 학습 — `train_teacher`

클래스 불균형(§4.5-1) 보정을 위해 `pos_weight` 를 데이터셋의 `class_weights()` 로부터 계산한다.

In [4]:
if not HAS_TORCH or train_ds is None:
    print("[skip] torch 미설치 또는 데이터셋 없음 — 학습을 건너뜁니다.")
else:
    from pm_safeline.utils.training import TrainConfig, train_teacher

    # class_weights() 는 [w_neg, w_pos] 정규화 가중치를 반환한다.
    # BCEWithLogitsLoss(pos_weight=...) 는 "양성 클래스에 곱할 배수" 이므로 w_pos/w_neg 비율 사용.
    base_ds = train_ds.dataset if hasattr(train_ds, "dataset") else train_ds
    weights = base_ds.class_weights()
    pos_weight = weights[1] / weights[0]
    print(f"pos_weight = {pos_weight.item():.4f}")

    cfg = TrainConfig(epochs=20, lr=1e-4, batch_size=16, patience=5)
    result = train_teacher(model, train_ds, valid_ds, cfg=cfg, pos_weight=pos_weight)
    print(f"best_epoch={result['best_epoch']} best_valid_auc={result['best_auc']:.4f}")


[skip] torch 미설치 또는 데이터셋 없음 — 학습을 건너뜁니다.


In [5]:
if not HAS_TORCH or train_ds is None:
    print("[skip] 학습 이력 시각화 건너뜀.")
else:
    import matplotlib.pyplot as plt

    history = result["history"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history["train_loss"], label="train_loss")
    axes[0].plot(history["valid_loss"], label="valid_loss")
    axes[0].set_title("loss"); axes[0].legend()
    axes[1].plot(history["valid_auc"], label="valid_auc", color="tab:green")
    axes[1].set_title("valid ROC-AUC"); axes[1].legend()
    plt.tight_layout()
    plt.show()


[skip] 학습 이력 시각화 건너뜀.


## 4. 신뢰도 추정 — `cross_validate` (k-fold)

데이터가 적으므로(§4.5-1) 고정 test 분할 대신 k-fold 교차검증으로 teacher 성능 신뢰도를 추정한다.

In [6]:
if not HAS_TORCH or train_ds is None:
    print("[skip] torch 미설치 또는 데이터셋 없음 — 교차검증을 건너뜁니다.")
else:
    from pm_safeline.utils.training import cross_validate
    from pm_safeline.datasets.roadview import PMRoadviewDataset, default_transform

    # k-fold 는 전체 manifest 기준으로 수행(train/valid 분할 이전 원본 Dataset 사용)
    full_ds = PMRoadviewDataset(transform=default_transform(train=True))

    def build_model_fn():
        return build_pm_risk_vit(PMRiskViTConfig(freeze_backbone=True))

    cv_cfg = TrainConfig(epochs=10, lr=1e-4, batch_size=16, patience=3)
    cv_result = cross_validate(build_model_fn, full_ds, n_splits=5, cfg=cv_cfg, pos_weight=pos_weight)
    print(f"fold AUCs: {cv_result['fold_aucs']}")
    print(f"mean AUC = {cv_result['mean_auc']:.4f} ± {cv_result['std_auc']:.4f}")


[skip] torch 미설치 또는 데이터셋 없음 — 교차검증을 건너뜁니다.


## 5. 보정 — `calibrate` (temperature scaling)

ViT teacher 는 소량 데이터에서 과신(overconfident)하기 쉽다(§4.5). valid set 으로 온도 T 를 학습해
확률 보정을 적용한다.

In [7]:
if not HAS_TORCH or train_ds is None:
    print("[skip] torch 미설치 또는 데이터셋 없음 — 보정을 건너뜁니다.")
else:
    from pm_safeline.utils.training import calibrate

    # best_state 로 복원 후 보정
    model.load_state_dict(result["best_state"])
    calibrator = calibrate(model, valid_ds, cfg, method="temperature")
    print(f"temperature = {calibrator.temperature:.4f}")


[skip] torch 미설치 또는 데이터셋 없음 — 보정을 건너뜁니다.


## 6. 체크포인트 저장 — `save_checkpoint`

In [8]:
if not HAS_TORCH or train_ds is None:
    print("[skip] torch 미설치 또는 데이터셋 없음 — 저장을 건너뜁니다.")
else:
    from pm_safeline.utils.training import save_checkpoint

    ckpt_path = Path("../../../checkpoints/pm_risk_vit_teacher.pt")
    save_checkpoint(
        model,
        ckpt_path,
        extra={
            "config": model.config.to_dict(),
            "temperature": calibrator.temperature,
            "best_valid_auc": result["best_auc"],
            "cv_mean_auc": cv_result["mean_auc"],
        },
    )


[skip] torch 미설치 또는 데이터셋 없음 — 저장을 건너뜁니다.
